# Ibà-ATLAS: fine-tune and benchmark N-ATLAS for fever triage

**Kaggle settings (right panel):** Accelerator = GPU T4 x2 (or P100), Internet = On, Secrets → add `HF_TOKEN` (a Hugging Face *write* token from the account that accepted the N-ATLaS licence).

Run the cells top to bottom. Total time ≈ 1.5–2.5 h. Generation is resumable: if the session dies, re-run the same cell.

In [ ]:
!pip install -q -U "transformers>=4.46" peft bitsandbytes accelerate huggingface_hub

In [ ]:
!git clone -q https://github.com/jemiridaniel/iba-natlas.git
%cd iba-natlas
!git log -1 --oneline

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Smoke test: 16 cases on base N-ATLAS (≈3 min, mostly the 16 GB download)

In [ ]:
!python -m natlas.generate --out results/smoke_base.jsonl --limit 16
!head -3 results/smoke_base.jsonl

## 2. Base N-ATLAS on the full 710-case benchmark

In [ ]:
!python -m natlas.generate --out results/preds_base.jsonl

## 3. QLoRA fine-tune (876 train / 97 val, 2 epochs, ≈30–45 min on T4)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m natlas.train --epochs 2 --batch 2 --accum 8

## 4. Fine-tuned model on the same benchmark

In [ ]:
!python -m natlas.generate --adapter outputs/iba-atlas-lora --out results/preds_ft.jsonl

## 5. Score and compare

In [ ]:
!python -m natlas.score results/preds_base.jsonl results/preds_ft.jsonl

## 6. Publish the adapter (private) and save results

Licence: derivatives keep the N-ATLaS Terms and the name must end with *Powered by Awarri*.

In [ ]:
!python -m natlas.merge_push --adapter outputs/iba-atlas-lora --repo jemiridaniel/Iba-ATLAS-8B-LoRA-Powered-by-Awarri --adapter-only

In [ ]:
!cp outputs/iba-atlas-lora/train_log.json results/
!zip -qr /kaggle/working/iba_atlas_results.zip results
print('Download iba_atlas_results.zip from the Output panel and send it to Claude.')